[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Errors and Exceptions


## What you will be able to do

Read a traceback and find the line that actually failed, catch the errors you expect without
hiding the ones you do not, and raise your own when your code is given something it cannot use.


## The idea

### The problem

Every notebook in this guide ends with errors run on purpose, and until now the message has
always been the end of the story: something broke, read the message, fix the code.

That works while you are the only person running it and the input is one line you typed. It
stops working as soon as the input comes from somewhere else.

A file might not exist. A number typed into a form might be `"twelve"`. A web service might
return nothing. None of those are bugs in your code, and none of them should stop a program
that processes a thousand records because one of them was odd. What you need is a way to say:
if this particular thing goes wrong, here is what to do instead.

### What an exception is

> An **exception** is an object Python creates when something goes wrong. Raising one interrupts
> the normal flow and passes the object up through the calling code until something handles it.
> If nothing does, the program stops and Python prints a **traceback**.
>
> A `try` block marks code that might fail. An `except` block says what to do when a particular
> kind of exception is raised inside it.

The word to notice is **particular**. Catching everything is easy and almost always wrong, for
reasons this notebook demonstrates rather than asserts.

### How to read a traceback

A traceback is printed with the newest information last, which is the opposite of how people
read. Three things matter, in this order:

1. **The last line.** It gives the exception type and the message. This is what went wrong.
2. **The last frame above it.** This is where it went wrong, with the file, line number and the
   line itself.
3. **The frames above that**, read upward. They show how execution arrived there, each one the
   call that led to the next.

The bottom of a traceback is the most specific place. Read it first, and read the earlier frames
only when the last one is not enough.

### Where you will meet this

Anywhere input comes from outside your program, which is most real programs. Reading files in
the **Files and Paths** notebook, calling web services in the **APIs and JSON** guide, and
loading data anywhere all raise exceptions as a matter of course rather than as a sign of
failure.

Reading tracebacks matters sooner than that. It is the main skill that separates being stuck
from being able to fix things, and it is worth practicing deliberately.

### What this notebook covers

- Reading a traceback with several frames, from the bottom up
- `try` and `except`, and catching one specific type
- Getting the message off the exception object with `as`
- Several `except` blocks, and why the order matters
- `else` and `finally`, and what each is for
- `raise`, for when your code is the one that has been given something wrong
- Writing your own exception type
- When not to catch anything at all
- Four errors, including the one that hides every other error you will ever make

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet:
read it, and read the output underneath it. Everything from Setup onward is where you start
running things, and the rest of the notebook takes this apart piece by piece.

```python
def load(raw):
    return parse(raw)

def parse(text):
    return int(text)

load("abc")
```

```
ValueError: invalid literal for int() with base 10: 'abc'
```

Read it from the bottom.

`ValueError: invalid literal for int() with base 10: 'abc'` is what went wrong. The frame
directly above it points at `return int(text)` inside `parse`, which is where. Above that,
`parse` was called by `load`, and `load` was called by the last line of the cell.

The bottom frame is the one to fix. The frames above answer "how did we get here" when that is
not obvious.


## Setup

Nothing to import.


In [2]:
# This notebook uses no libraries.
print("Ready.")


Ready.


## Worked examples

### try and except

`try` holds the code that might fail. `except` runs only if it does.


In [3]:
def to_number(text):
    try:
        return int(text)
    except ValueError:
        return None

print(to_number("42"))
print(to_number("abc"))


42
None


The second call did not stop the program. `int("abc")` raised `ValueError`, the `except` block
caught it, and the function returned `None` instead.

Naming the type matters. `except ValueError:` catches that and nothing else, so a different
problem still surfaces as an error rather than being turned into `None`.

### Reading the message

`as` gives you the exception object, and printing it gives the message.


In [4]:
def to_number(text):
    try:
        return int(text)
    except ValueError as e:
        print("could not convert:", e)
        return None

to_number("abc")


could not convert: invalid literal for int() with base 10: 'abc'


The message is the same text the traceback would have shown. `type(e).__name__` gives the type
if you need to report it.


In [5]:
try:
    int("abc")
except ValueError as e:
    print("type:   ", type(e).__name__)
    print("message:", e)


type:    ValueError
message: invalid literal for int() with base 10: 'abc'


### More than one kind of failure

A `try` can have several `except` blocks. Python uses the first one that matches.


In [6]:
def divide(text, divisor):
    try:
        return int(text) / divisor
    except ValueError:
        return "that is not a number"
    except ZeroDivisionError:
        return "cannot divide by zero"

print(divide("10", 2))
print(divide("abc", 2))
print(divide("10", 0))


5.0
that is not a number
cannot divide by zero


### Order matters, because exceptions have families

Exception types form a hierarchy. `ZeroDivisionError` is a kind of `ArithmeticError`, which is a
kind of `Exception`.


In [7]:
for exc in (ZeroDivisionError, ValueError, KeyError, IndexError):
    family = " -> ".join(c.__name__ for c in exc.__mro__[:3])
    print(f"{exc.__name__:<20} {family}")


ZeroDivisionError    ZeroDivisionError -> ArithmeticError -> Exception
ValueError           ValueError -> Exception -> BaseException
KeyError             KeyError -> LookupError -> Exception
IndexError           IndexError -> LookupError -> Exception


An `except` block catches its type **and everything below it**. So a general type placed first
catches everything, and the specific blocks after it never run.


In [8]:
def classify(fn):
    try:
        fn()
    except Exception:
        return "something went wrong"
    except ZeroDivisionError:
        return "divided by zero"      # unreachable

print(classify(lambda: 1 / 0))


something went wrong


`Exception` caught it first, so the specific block never had a chance. This is the same
unreachable-branch problem the **Conditionals** notebook showed with `if` and `elif`, and it has
the same fix: **most specific first**.


In [9]:
def classify(fn):
    try:
        fn()
    except ZeroDivisionError:
        return "divided by zero"
    except Exception:
        return "something went wrong"

print(classify(lambda: 1 / 0))
print(classify(lambda: int("x")))


divided by zero
something went wrong


### else and finally

`else` runs when the `try` block did **not** raise. `finally` runs either way.


In [10]:
def report(text):
    try:
        value = int(text)
    except ValueError:
        print("  could not read a number")
    else:
        print("  got", value)
    finally:
        print("  done")

print("report('5')")
report("5")
print("report('x')")
report("x")


report('5')
  got 5
  done
report('x')
  could not read a number
  done


`else` keeps the `try` block down to the one line that might fail. Everything that should happen
only on success goes in `else`, where it cannot accidentally be protected by the `except`.

`finally` is for cleanup that has to happen regardless: closing a file, releasing a connection.
The **Files and Paths** notebook shows the `with` statement, which handles the common case
without needing `finally` at all.


### Raising your own

When your code is handed something it cannot work with, say so rather than continuing.


In [11]:
def set_temperature(celsius):
    if celsius < -273.15:
        raise ValueError(f"{celsius} is below absolute zero")
    return celsius

print(set_temperature(20))
print(set_temperature(-300))


20


ValueError: -300 is below absolute zero

`raise` stops the function immediately, like `return`, but signals a failure instead of handing
back a value. The message should say what was wrong and, where possible, what was received.

Failing early is usually better than returning something that looks valid. A function that
quietly returns `None` for bad input moves the error to whichever line uses that `None`, which
may be a long way away.

### Your own exception type

For anything beyond a small script, a specific type lets callers catch your errors without
catching everything.


In [12]:
class SensorError(Exception):
    """Raised when a sensor reading is outside its valid range."""

def read_sensor(value):
    if not -50 <= value <= 150:
        raise SensorError(f"reading {value} is outside the range -50 to 150")
    return value

try:
    read_sensor(999)
except SensorError as e:
    print("handled:", e)


handled: reading 999 is outside the range -50 to 150


`class SensorError(Exception):` is enough. The **Object-Oriented Python** guide covers classes
properly; for now this line is the whole recipe.

The gain is precision at the call site. `except SensorError:` catches sensor problems and
nothing else, so a typo in the same block still reaches you as a traceback.


### When not to catch

Catching an exception is a decision to continue. Only make it when you know what to do instead.


In [13]:
readings = ["21", "22", "not recorded", "24"]

total = 0
used = 0
for r in readings:
    try:
        total += int(r)
        used += 1
    except ValueError:
        continue          # a known, expected gap in the data

print(f"{used} of {len(readings)} readings, mean {total / used:.1f}")


3 of 4 readings, mean 22.3


That is a good catch: the bad value is expected, skipping it is correct, and the count reports
how many were used.

A catch that returns a default nobody asked for, or prints a message and carries on with wrong
data, is worse than the crash it replaced. If you cannot say what should happen instead, let it
raise.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/16-errors-and-exceptions-solutions.ipynb).

**1.** Write `safe_int(text)` that returns the number, or `None` if the text is not a number.
Print it for `"7"` and for `"seven"`.


In [14]:
# your code here


**2.** Rewrite it to print the exception message before returning `None`, using `as`.


In [15]:
# your code here


**3.** Write a function that divides two numbers and handles both a non-numeric input and a
zero divisor, with a different message for each.


In [16]:
# your code here


**4.** Given `ages = {"ada": 36}`, write code that looks up `"grace"` and prints
`"not on file"` instead of raising. Catch `KeyError`.


In [17]:
# your code here


**5.** Write `withdraw(balance, amount)` that raises `ValueError` when the amount is more than
the balance, and returns the new balance otherwise. Show both outcomes.


In [18]:
# your code here


**6.** Given `values = ["3", "4", "x", "5"]`, total the ones that are numbers and print how
many were skipped.


In [19]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### The bare except, which hides everything

This is the most damaging habit in this notebook, and it does not raise.


In [20]:
def broken(text):
    try:
        cleaned = text.strip()
        return int(cleand)         # typo: cleand
    except:
        return None

print(broken("42"))


None


`None`, and the input was perfectly valid.

The typo raised `NameError`, the bare `except` caught it along with everything else, and the
function reported failure. Nothing tells you a typo exists. This is how a bug survives for
months.

Name the type you expect and the typo surfaces immediately:


In [21]:
def broken(text):
    try:
        cleaned = text.strip()
        return int(cleand)
    except ValueError:
        return None

print(broken("42"))


NameError: name 'cleand' is not defined

`name 'cleand' is not defined`, which is the message you needed. **Never write a bare
`except:`.** If you genuinely mean any error, write `except Exception as e:` and at minimum log
`e`.


### KeyError: the message is the key


In [22]:
ages = {"ada": 36, "alan": 41}
print(ages["grace"])


KeyError: 'grace'

`KeyError: 'grace'` gives the key and nothing else, which is usually enough. Catch it, or use
`.get()` from the **Dictionaries** notebook when a miss is expected.


In [23]:
try:
    print(ages["grace"])
except KeyError as e:
    print("no entry for", e)


no entry for 'grace'


### The try block that is too wide


In [24]:
values = ["3", "4", "x"]

try:
    numbers = [int(v) for v in values]
    total = sum(numbers)
    mean = total / len(numbers)
    print(mean)
except ValueError:
    print("could not read the numbers")


could not read the numbers


The message is printed, which looks like it worked. But four separate operations were inside
that `try`, and only one of them was expected to fail. If `len(numbers)` had been zero, the
`ZeroDivisionError` would not have been caught here at all, and if the conversion succeeded but
`sum` failed, the message would be wrong.

Keep the `try` around the line that can fail, and put the rest in `else`:


In [25]:
try:
    numbers = [int(v) for v in values]
except ValueError:
    print("could not read the numbers")
else:
    print(sum(numbers) / len(numbers))


could not read the numbers


### Catching an exception the code cannot raise


In [26]:
try:
    result = 10 / 2
except IndexError:
    print("index problem")

print(result)


5.0


No error, and no protection either. `10 / 2` cannot raise `IndexError`, so that `except` block
can never run. It reads as though the risk was handled.

This is worth checking whenever you copy an error handler from elsewhere: the type has to match
what the code inside can actually raise.


## Recap

- Read a traceback from the **bottom**: type and message, then the line, then how you got there.
- `try` marks code that might fail; `except TypeName:` says what to do when it does.
- `as e` gives the exception object, and printing it gives the message.
- Exception types form families, so put the **most specific** `except` first.
- `else` runs when nothing was raised; `finally` runs either way.
- `raise` signals failure from your own code, with a message saying what was wrong.
- `class MyError(Exception):` defines a type callers can catch precisely.
- **Never write a bare `except:`.** It hides typos and every other unrelated failure.
- Only catch when you know what to do instead. Otherwise let it raise.


## What is next

The **Files and Paths** notebook, which reads and writes real files. It is the first place
input comes from outside the notebook, which makes it the first place the handling in this
notebook is not optional.


---

&#8592; **Previous:** [Scope](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/15-scope.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)  &nbsp;·&nbsp;  **Next:** [Files and Paths](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/17-files-and-paths.ipynb) &#8594;
